In [1]:
# -*- coding: utf-8 -*-
from turtle import title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os
from ansys.aedt.core import generate_unique_name
from ansys.aedt.core import Desktop, Hfss, constants
from ansys.aedt.core.modeler.cad.object_3d import Object3d
from ansys.aedt.core.application.analysis import Analysis

In [2]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "False"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)

w0=100
l0=150
w=3
h=1.65

hfss = Hfss(solution_type="Modal")
hfss.modeler.model_units = "mm"

p = hfss.modeler.primitives


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_24d2afcf-97b6-4791-933b-93b4e4657977.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 50387.
PyAEDT INFO: Electronics Desktop started on gRPC port: 50387 after 28.874329090118408 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 3104.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT

In [3]:
gnd = p.create_rectangle(
    constants.AXIS.Z,
    origin= (-w0/2, -l0/2,0) ,
    sizes= (w0, l0),
    name="ground"
)

substrate = p.create_box(
    [-w0/2,-l0/2,0],
    [w0,l0,h],
    material="FR4_epoxy",
    name="substrate"
)

path = p.create_rectangle(
    constants.AXIS.Z,
    origin=(-w/2,-l0/2, h) ,
    sizes=(w,l0),
    name="path"
)

port1 = p.create_rectangle(
    constants.AXIS.Y,
    origin=(-w/2,l0/2,0) ,
    sizes=(h,w),
    name="port1"
)

port2 = p.create_rectangle(
    constants.AXIS.Y,
    origin=(-w/2,-l0/2,0) ,
    sizes=(h,w),
    name="port2"
)

air = p.create_region(
    20,
    pad_type="Absolute Offset",
    name="airbox"    
)

hfss.assign_radiation_boundary_to_objects("airbox")
hfss.change_material_override()

PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Boundary Radiation Rad__EKRZ1U has been created.
PyAEDT INFO: Enabling Material Override


True

In [4]:
hfss.lumped_port(
    assignment=port1.name,                 
    reference=port1,
    integration_line=hfss.AxisDir.ZPos,    
    impedance=50,
    name=generate_unique_name("port1"),
    renormalize=False,
    terminals_rename=False
)

hfss.lumped_port(
    assignment=port2.name,                 
    reference=port2,
    integration_line=hfss.AxisDir.ZPos,    
    impedance=50,
    name=generate_unique_name("port2"),
    renormalize=False,
    terminals_rename=False
)

hfss.assign_perfect_e(
    gnd.name
)

hfss.assign_perfect_e(
    path.name
)

PyAEDT INFO: Boundary Lumped Port port1_FFDDJD has been created.
PyAEDT INFO: Boundary Lumped Port port2_7T4BG4 has been created.
PyAEDT INFO: Boundary Perfect E PerfectE_ground has been created.
PyAEDT INFO: Boundary Perfect E PerfectE_path has been created.


In [5]:
my_plot =hfss.plot(show=False, plot_air_objects=False)
my_plot.show_axes = False
my_plot.show_grid = False
my_plot.isometric_view = True
my_plot.plot(
    os.path.join(hfss.working_directory, "MicroStrip.jpg")    
)

setup = hfss.create_setup("MySetup")

PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec


Widget(value='<iframe src="http://localhost:50422/index.html?ui=P_0x1825bc32bc0_0&reconnect=auto" class="pyvis…

In [6]:
def SingleFrequency(
    freq,
    maxPasses,
    maxDeltaS= 0.02,
    name="MySetup"
):
    hfss.create_open_region(frequency=freq)
    setup.props["Frequency"] = freq
    setup.props["MaximumPasses"] = maxPasses
    setup.props["MaxDeltaS"] =maxDeltaS
    setup.props["UseIterativeSolver"] = True
    setup.props["SaveRadFieldsOnly"] = True

SingleFrequency("5GHz", 6,0.02)

PyAEDT INFO: Open Region correctly created.
PyAEDT INFO: Project Project14 Saved correctly


In [7]:
def LinearStepSweep(
    fStart,
    fStop,
    stepSize=0.0125,
    units="GHz",
    saveFields=True,
    saveRadFields=True
):
    hfss.create_linear_step_sweep(
        setup=setup.name,
        unit=units,
        start_frequency=fStart,
        stop_frequency=fStop,
        step_size = stepSize,
        name= "sweep",
        sweep_type="Interpolating",
        save_fields=saveFields,
        save_rad_fields=saveRadFields
    )

LinearStepSweep(1.0,5.0)


PyAEDT INFO: Parsing C:/Users/luis.costa/Documents/Ansoft/Project14.aedt.
PyAEDT INFO: File C:/Users/luis.costa/Documents/Ansoft/Project14.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.09964823722839355
PyAEDT INFO: Linear step sweep sweep has been correctly created


In [8]:

def SaveAnalyseSetup(
    nameFolder= "MicroStrip.aedt",
    setup="MySetup"
):
    hfss.save_project(os.path.join(temp_folder, nameFolder))
    hfss.analyze_setup(setup)


SaveAnalyseSetup("MicroStrip.aedt")


PyAEDT INFO: Project MicroStrip Saved correctly
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/HFSS correctly changed.
PyAEDT INFO: Solving design setup MySetup
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/HFSS correctly changed.
PyAEDT INFO: Design setup MySetup solved correctly in 0.0h 1.0m 56.0s


In [9]:
d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True